In [1]:
import numpy as np
import scipy

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

import tifffile
import fastplotlib as fpl

To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

No windowing system present. Using surfaceless platform
No config found!
No config found!


Available devices:
✅ (default) | Tesla V100-SXM2-32GB | DiscreteGPU | Vulkan | 550.90.07
❗ | Tesla V100-SXM2-32GB/PCIe/SSE2 | Unknown | OpenGL | 


In [2]:
from sklearn.utils.extmath import randomized_svd

In [6]:
from time import time

In [8]:
from scipy.sparse.linalg import svds

In [3]:
Hp = np.load("./Hp_dendrites.npy")

In [4]:
Hp.shape

(36, 54700816)

In [39]:
k = 10_000_000
Hp_ = Hp[:, :k]
rank = 1

In [94]:
%%time
u_rand, s_rand, v_rand = randomized_svd(Hp_, n_components=rank)

CPU times: user 31.3 s, sys: 3.95 s, total: 35.3 s
Wall time: 35.1 s


In [95]:
np.linalg.norm(Hp_ - u_rand @ np.diag(s_rand) @ v_rand, ord="fro")

96567141.3075258

In [96]:
%%time
u_s, s_s, v_s = svds(Hp_, k=rank)

CPU times: user 40.5 s, sys: 20.9 s, total: 1min 1s
Wall time: 1min 1s


In [97]:
np.linalg.norm(Hp_ - u_s @ np.diag(s_s) @ v_s, ord="fro")

96567141.30752599

In [44]:
from dask_ml.decomposition import TruncatedSVD

In [53]:
from dask.array.linalg import svd_compressed

In [45]:
import dask.array as da

In [73]:
a = da.asarray(Hp_, chunks=(6, int(k / 100)))

In [74]:
a

dask.array<array, shape=(36, 10000000), dtype=float64, chunksize=(6, 100000), chunktype=numpy.ndarray>

In [51]:
svd = TruncatedSVD(n_components=1)

In [52]:
%%time
svd.fit(a)

MemoryError: Unable to allocate 728. TiB for an array with shape (10000000, 10000000) and data type float64

In [75]:
%%time
u_dask, s_dask, v_dask = svd_compressed(a, k=1)

CPU times: user 25.1 ms, sys: 4.33 ms, total: 29.5 ms
Wall time: 27.8 ms


In [76]:
%%time
v_computed = v_dask.compute()

CPU times: user 31.6 s, sys: 3.26 s, total: 34.9 s
Wall time: 7.65 s


In [91]:
a_ = u_dask @ dask.array.diag(s_dask) @ v_dask

In [92]:
a__ = a_.compute()

In [93]:
dask.array.linalg.norm(a - (u_dask @ dask.array.diag(s_dask) @ v_dask), ord="fro").compute()

96570226.4982952

In [98]:
v_s

array([[-1.28076929e-05, -1.12540944e-05, -1.16915784e-05, ...,
        -7.40121390e-05, -7.30062309e-05, -7.97921304e-05]])

In [99]:
v_computed

array([[1.28076929e-05, 1.12540944e-05, 1.16915785e-05, ...,
        7.40121390e-05, 7.30062310e-05, 7.97921304e-05]])

In [102]:
np.linalg.norm(v_s + v_computed)

2.185574103610926e-10